[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Create, Read and Update Models &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup and the family its worked examples wrote: `HeroBase`, `Hero`,
`HeroCreate`, `HeroPublic`, `HeroUpdate` and `Team`, with the eight heroes and three teams loaded.
Run it first. The tasks follow one another, and the last cell removes the scratch folder.


In [1]:
import contextlib
import re
import shutil
import subprocess
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from pydantic import ValidationError
from sqlalchemy import event, insert
from sqlalchemy.exc import IntegrityError
from sqlmodel import Field, Session, SQLModel, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


@contextlib.contextmanager
def catching():
    """Collects warnings instead of printing them: Python prints one with the file that raised it."""
    with warnings.catch_warnings(record=True) as raised:
        warnings.simplefilter("always")
        yield raised


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

with Session(engine) as session:
    print("sqlmodel", sqlmodel.__version__, "|", len(session.exec(select(Hero)).all()), "heroes loaded")


with catching() as warned:                                          # the class names are used a second time
    SQLModel.metadata.clear()

    class HeroBase(SQLModel):
        """What every hero has, wherever it is going. No table."""

        name: str = Field(index=True, max_length=50)
        age: int | None = Field(default=None, index=True)
        team_id: int | None = Field(default=None, foreign_key="team.id")


    class Hero(HeroBase, table=True):
        """The row: the shared fields, the id the database gives, and the secret."""

        id: int | None = Field(default=None, primary_key=True)
        secret_name: str = Field(max_length=60)


    class HeroCreate(HeroBase):
        """What a caller may send to make a hero: no id, and the secret it must supply."""

        secret_name: str = Field(max_length=60)


    class HeroPublic(HeroBase):
        """What may be sent back: the shared fields and the id, and no secret at all."""

        id: int


    class HeroUpdate(SQLModel):
        """A change: every field optional, so that anything left out means leave it alone."""

        name: str | None = None
        age: int | None = None
        team_id: int | None = None
        secret_name: str | None = None


    class Team(SQLModel, table=True):
        id: int | None = Field(default=None, primary_key=True)
        name: str = Field(index=True, max_length=50)
        headquarters: str = Field(max_length=60)


sqlmodel 0.0.42 | 8 heroes loaded


**1.** The same family, for a team.


In [2]:
class TeamBase(SQLModel):
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)


class TeamCreate(TeamBase):
    pass


class TeamPublic(TeamBase):
    id: int


for model in (TeamBase, TeamCreate, TeamPublic):
    print(f"  {model.__name__:<11} {list(model.model_fields)}")


  TeamBase    ['name', 'headquarters']
  TeamCreate  ['name', 'headquarters']
  TeamPublic  ['name', 'headquarters', 'id']


`TeamCreate` adds nothing, since a team has no secret and no field a caller must supply beyond the
shared two. It still earns its place: it is the class that says a caller may not send an id.


**2.** A team created and shown, with no field named by hand.


In [3]:
arriving = TeamCreate(name="Grand Council", headquarters="Grand Hall")
team = Team.model_validate(arriving)
with Session(engine, expire_on_commit=False) as session:
    session.add(team)
    session.commit()

print("created:", TeamPublic.model_validate(team).model_dump())


created: {'name': 'Grand Council', 'headquarters': 'Grand Hall', 'id': 4}


Three calls, and the id came from the database rather than from the caller.


**3.** A change, and the two dumps.


In [4]:
class TeamUpdate(SQLModel):
    name: str | None = None
    headquarters: str | None = None


change = TeamUpdate(headquarters="The Old Vault")
print("every field      :", change.model_dump())
print("only what was set:", change.model_dump(exclude_unset=True))


every field      : {'name': None, 'headquarters': 'The Old Vault'}
only what was set: {'headquarters': 'The Old Vault'}


The first would set the name to `None`; the second mentions the one field the caller sent.


**4.** The same change applied both ways.


In [5]:
with Session(engine) as session:
    council = session.get(Team, 4)
    council.sqlmodel_update(change.model_dump(exclude_unset=True))
    session.add(council)
    session.commit()
    session.refresh(council)
    print("with exclude_unset:", fields(council))

    preventers = session.get(Team, 1)
    preventers.sqlmodel_update(TeamUpdate(headquarters="Sharp Tower").model_dump())
    try:
        session.commit()
    except IntegrityError as error:
        print("with the full dump:", str(error).splitlines()[0])
    session.rollback()
    print("and the team is unchanged:", fields(session.get(Team, 1)))


with exclude_unset: {'id': 4, 'name': 'Grand Council', 'headquarters': 'The Old Vault'}
with the full dump: (sqlite3.IntegrityError) NOT NULL constraint failed: team.name
and the team is unchanged: {'id': 1, 'name': 'Preventers', 'headquarters': 'Sharp Tower'}


The first kept the name and changed the headquarters. The second set the name to `None`, and the
column is `NOT NULL`, so the database refused the update and the rollback left the team as it was.


**5.** A list model, for the heroes of one team.


In [6]:
class HeroListed(SQLModel):
    id: int
    name: str


with Session(engine) as session:
    listed = [HeroListed.model_validate(hero)
              for hero in session.exec(select(Hero).where(Hero.team_id == 2).order_by(Hero.name))]
for hero in listed:
    print(" ", hero.model_dump())


  {'id': 5, 'name': 'Black Lion'}
  {'id': 1, 'name': 'Deadpond'}
  {'id': 6, 'name': 'Dr. Weird'}


Two fields, and the secret name is not one of them. A list model is usually smaller than a public
model, because a list is read far more often than a page.


**6.** A team created from raw data, or refused.


In [7]:
def create_team(engine, raw):
    """Check what arrived, write the team, and return what may be shown."""
    try:
        arriving = TeamCreate.model_validate(raw)
    except ValidationError as error:
        return {"refused": message(error).splitlines()[1:3]}
    team = Team.model_validate(arriving)
    with Session(engine, expire_on_commit=False) as session:
        session.add(team)
        session.commit()
    return TeamPublic.model_validate(team).model_dump()


print(create_team(engine, {"name": "Night Watch", "headquarters": "The Bell Tower"}))
print(create_team(engine, {"name": "Night Watch"}))


{'name': 'Night Watch', 'headquarters': 'The Bell Tower', 'id': 5}
{'refused': ['headquarters', "  Field required [type=missing, input_value={'name': 'Night Watch'}, input_type=dict]"]}


The second has no headquarters, so `TeamCreate` refused it, naming the field and saying what was
wrong with it, and no database was asked anything.

Last, the engine lets go of the file, and this cell removes the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Create, Read and Update Models](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/07-create-read-and-update-models.ipynb)  &nbsp;&middot;&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
